# 🎓 TeachRL — Training Notebook

**Theme 4: Self-Improvement** | Meta PyTorch Hackathon x Scaler 2025

Trains a PPO agent on TeachRL: an adaptive tutoring environment with 8 hidden student archetypes and self-play escalation.

**Links:** [HF Space](https://huggingface.co/spaces/ArchedEquation/TeachRL) | [GitHub](https://github.com/ArchedEquation/TeachRL)

---

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install stable-baselines3 gymnasium pydantic fastapi uvicorn pyyaml openai matplotlib torch

## Step 2 — Clone Repository

In [ ]:
import os

REPO_DIR = '/content/TeachRL'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ArchedEquation/TeachRL {REPO_DIR}
else:
    print('Repo already cloned')

os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 3 — Verify Environment Works

In [ ]:
import os, sys
os.chdir('/content/TeachRL'); sys.path.insert(0, '/content/TeachRL')

from env.environment import TeachRLEnv, TASK_REGISTRY
from env.archetypes import ALL_ARCHETYPES

print(f'Tasks: {list(TASK_REGISTRY.keys())}')
print(f'Archetypes: {len(ALL_ARCHETYPES)}')

env = TeachRLEnv(task_id='blind_teaching', seed=42, eval_mode=True)
obs = env.reset()
result = env.step({'concept':'algebra_basics','difficulty':'medium','hint_given':False,'archetype_guess':None})
print(f'Step OK — reward={result.reward:.3f}')
print(f'True archetype: {env._sim.archetype_id.value}')
print(f'Expert hint: {obs.expert_hint[:80]}')

## Step 4 — Run Baseline Evaluation (Before Training)

In [ ]:
os.chdir('/content/TeachRL'); sys.path.insert(0, '/content/TeachRL')
!python baseline/baseline_inference.py --episodes 5 --seed 42

## Step 5 — Train Archetype Classifier

Trains a supervised neural net to identify student archetypes from observable signals.
Achieves ~92% validation accuracy.

In [ ]:
os.chdir('/content/TeachRL'); sys.path.insert(0, '/content/TeachRL')

from baseline.archetype_classifier import train_classifier
model = train_classifier(n_episodes=4000, epochs=80, seed=42)
print('Classifier training complete!')

## Step 6 — Train PPO — Easy Task (Archetype Identification)

In [ ]:
os.chdir('/content/TeachRL')
!python baseline/rl_agent.py --train --task archetype_identification

## Step 7 — Train PPO — Medium Task (Adaptive Curriculum)

In [ ]:
os.chdir('/content/TeachRL')
!python baseline/rl_agent.py --train --task adaptive_curriculum

## Step 8 — Train PPO — Hard Task (Blind Teaching)

In [ ]:
os.chdir('/content/TeachRL')
!python baseline/rl_agent.py --train --task blind_teaching

## Step 9 — Train PPO — Expert Task (Self-Play Escalation)

**This is the Theme 4 core task.** The environment gets harder as the agent improves.

In [ ]:
os.chdir('/content/TeachRL')
!python baseline/rl_agent.py --train --task self_play_escalation

## Step 10 — Full Evaluation — PPO + Classifier vs All Baselines

In [ ]:
os.chdir('/content/TeachRL')
!python baseline/rl_agent.py --eval --task all --episodes 10

## Step 11 — Display Training Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

os.chdir('/content/TeachRL')
plots = sorted(glob.glob('training_plots/*.png'))
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, path in zip(axes.flat, plots[:4]):
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(path.split('/')[-1].replace('.png','').replace('_',' ').title(), fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('training_plots/all_plots_combined.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 12 — Demo: Watch the Trained Agent Teach

In [ ]:
import os, sys
os.chdir('/content/TeachRL'); sys.path.insert(0, '/content/TeachRL')

from stable_baselines3 import PPO
from env.environment import TeachRLEnv, TASK_REGISTRY
from env.gym_wrapper import obs_to_vector, int_to_action
from baseline.archetype_classifier import ArchetypeClassifier

task_id   = 'blind_teaching'
max_steps = TASK_REGISTRY[task_id]['max_steps']
model     = PPO.load(f'models/ppo_{task_id}')
clf       = ArchetypeClassifier()

def agent(obs_dict):
    from env.gym_wrapper import obs_to_vector, int_to_action
    import numpy as np
    vec = obs_to_vector(obs_dict, max_steps)
    action, _ = model.predict(vec, deterministic=True)
    c, d = int_to_action(int(action))
    guess = clf.predict_from_obs(obs_dict) if clf.is_loaded else None
    return {'concept': c, 'difficulty': d, 'hint_given': False, 'archetype_guess': guess}

env = TeachRLEnv(task_id=task_id, seed=77, eval_mode=True)
obs = env.reset(seed=77)
print(f'True archetype: {env._sim.archetype_id.value}')
print(f'Expert hint: {obs.expert_hint}')
print()

done = False
while not done:
    action = agent(obs.model_dump())
    result = env.step(action)
    obs    = result.observation
    done   = result.done

print(env.render())
print(f'Final Score: {env._task_score():.4f}')

## Step 13 — Test the Live API

In [ ]:
import requests

BASE = 'https://archedequation-teachrl.hf.space'

r = requests.get(f'{BASE}/')
print('Status:', r.json()['status'])
print('Tasks:', r.json()['tasks'])

r   = requests.post(f'{BASE}/reset', json={'task_id':'blind_teaching','seed':42})
sid = r.json()['session_id']
print(f'Session: {sid[:8]}...')

r = requests.post(f'{BASE}/step',
    json={'session_id':sid,'concept':'algebra_basics','difficulty':'medium',
          'archetype_guess':'anxious_perfectionist'})
print(f'Reward: {r.json()["reward"]:.3f}  Done: {r.json()["done"]}')